# Statystyka dla Data Analyst/Engineer — Moduł 5: Próbkowanie i Centralne Twierdzenie Graniczne

To jeden z najważniejszych modułów całego kursu. Do tej pory pracowaliśmy na jednym,
konkretnym zbiorze 400 klientów, tak jakby to była cała nasza wiedza o świecie. W
praktyce Twoje 400 klientów to zwykle **próbka** wyciągnięta z dużo większej
**populacji** (wszystkich obecnych i przyszłych klientów). Statystyka wnioskująca —
którą zaczynamy tu na dobre — odpowiada na pytanie: co mogę powiedzieć o CAŁEJ populacji,
mając tylko próbkę?

## Spis treści
1. [Populacja kontra próbka, parametr kontra statystyka](#sec1)
2. [Rozkład próbkowania średniej](#sec2)
3. [Centralne Twierdzenie Graniczne — symulacja](#sec3)
4. [Błąd standardowy (standard error)](#sec4)
5. [Jak wielkość próbki wpływa na precyzję](#sec5)
6. [Podsumowanie i ćwiczenia](#sec6)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

W tym module będziemy traktować nasz zbiór 400 klientów jako **całą populację** (znamy jej prawdziwą, dokładną średnią), żeby móc symulować losowe próbkowanie z niej i porównywać próbki z prawdą, którą normalnie byśmy nie znali.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Populacja kontra próbka, parametr kontra statystyka

- **Populacja** — cały zbiór, o który nam chodzi (np. wszyscy obecni i przyszli klienci
  sklepu). Liczbę opisującą populację (np. jej prawdziwą średnią) nazywamy
  **parametrem**, tradycyjnie oznaczanym grecką literą, np. `mu` (średnia populacji).
- **Próbka** — podzbiór populacji, który faktycznie zmierzyliśmy/zebraliśmy (np. nasze
  400 klientów). Liczbę policzoną z próbki (np. jej średnią) nazywamy **statystyką**,
  oznaczaną łacińską literą, np. `x-bar` (średnia próbki).

Kluczowe pytanie statystyki wnioskującej: jak dobrze statystyka z próbki (`x-bar`)
przybliża nieznany parametr populacji (`mu`)? Odpowiedź zależy od tego, jak próbka została
pobrana i jak duża jest.

In [ ]:
mu_populacji = klienci["wydatki_miesieczne"].mean()  # w praktyce tej liczby NIE znasz
print(f"Prawdziwa srednia calej 'populacji' (400 klientow): {mu_populacji:.2f}")


> ⚠️ **Losowanie musi być reprezentatywne**
>
> Cała matematyka w tym module zakłada **losowe, reprezentatywne próbkowanie** — każdy element populacji ma znaną szansę trafienia do próbki. Jeśli próbka jest obciążona (np. ankietujesz tylko klientów, którzy zostawili opinię, albo tylko z jednego miasta), żadna z poniższych technik tego nie naprawi — "garbage in, garbage out". To zagadnienie samo w sobie (błąd doboru próby) wykracza poza ten kurs, ale warto o nim pamiętać zawsze, gdy zbierasz dane.

<a id="sec2"></a>
## 2. Rozkład próbkowania średniej

Wyobraź sobie, że zamiast brać JEDNĄ próbkę 30 klientów, bierzesz tysiące różnych próbek
30-osobowych (za każdym razem losując od nowa) i za każdym razem liczysz średnią. Dostaniesz
tysiące różnych średnich próbkowych — same te średnie też mają swój rozkład! Nazywamy go
**rozkładem próbkowania średniej** (*sampling distribution of the mean*).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
wielkosc_probki = 30
liczba_probek = 5000

srednie_probkowe = [
    rng.choice(klienci["wydatki_miesieczne"], size=wielkosc_probki, replace=False).mean()
    for _ in range(liczba_probek)
]

plt.figure(figsize=(8, 4))
plt.hist(srednie_probkowe, bins=40, edgecolor="white")
plt.axvline(mu_populacji, color="red", linestyle="--", label=f"Prawdziwa srednia populacji: {mu_populacji:.1f}")
plt.xlabel("Srednia probki (n=30)")
plt.ylabel("Liczba probek")
plt.title(f"Rozklad probkowania sredniej ({liczba_probek} probek po {wielkosc_probki} klientow)")
plt.legend()
plt.show()


Zauważ dwie rzeczy: (1) rozkład średnich próbkowych jest wyraźnie **skupiony wokół
prawdziwej średniej populacji** — nawet jeśli pojedyncza próbka czasem trafi daleko od
prawdy, w długim przebiegu średnie się "wyśrodkowują"; (2) kształt tego rozkładu wygląda
**w przybliżeniu jak dzwon (rozkład normalny)** — mimo że oryginalne `wydatki_miesieczne`
(zobacz Moduł 2) są wyraźnie prawoskośne, wcale niepodobne do dzwonu!

<a id="sec3"></a>
## 3. Centralne Twierdzenie Graniczne — symulacja

To, co właśnie zaobserwowaliśmy, to nie przypadek — to **Centralne Twierdzenie Graniczne**
(CTG, *Central Limit Theorem*): niezależnie od kształtu rozkładu populacji, rozkład
średnich z wystarczająco licznych próbek zbliża się do rozkładu normalnego, gdy wielkość
próbki (`n`) rośnie. To jedno z najważniejszych twierdzeń w całej statystyce — bez niego
większość testów statystycznych (Moduły 7+) po prostu by nie działała.

Zobaczmy to na jeszcze bardziej skrajnym przykładzie niż wydatki: wylosujmy dane z
rozkładu wykładniczego (jeszcze mocniej skośnego) i porównajmy kształt rozkładu
próbkowania dla różnych wielkości próbki `n`.

In [ ]:
from scipy import stats

# populacja "z definicji": silnie prawoskosny rozklad wykladniczy
populacja_wykladnicza = stats.expon(scale=100).rvs(size=100_000, random_state=1)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

axes[0].hist(populacja_wykladnicza, bins=50, color="gray")
axes[0].set_title("Populacja (rozklad wykladniczy)")

for ax, n in zip(axes[1:], [2, 10, 50]):
    srednie = [
        rng.choice(populacja_wykladnicza, size=n, replace=False).mean()
        for _ in range(3000)
    ]
    ax.hist(srednie, bins=40, color="steelblue")
    ax.set_title(f"Srednie probek, n={n}")

plt.tight_layout()
plt.show()


Populacja (skrajnie po lewej) jest bardzo skośna. Przy `n=2` rozkład średnich jest
wciąż wyraźnie skośny — dwie obserwacje to za mało, żeby "wygładzić" asymetrię. Przy
`n=10` kształt już wyraźnie przypomina dzwon. Przy `n=50` jest niemal idealnie
symetryczny i dzwonowaty — **CTG w akcji**.

> 📊 **Jak duże n wystarczy?**
>
> Popularna reguła kciuka mówi, że `n >= 30` zwykle wystarcza, żeby CTG "zadziałało" wystarczająco dobrze dla większości rozkładów spotykanych w praktyce. Im bardziej skośna/nietypowa populacja (jak nasz rozkład wykładniczy), tym większe `n` może być potrzebne. Jeśli populacja sama jest już w miarę normalna, wystarczy nawet bardzo małe `n`.

<a id="sec4"></a>
## 4. Błąd standardowy (standard error)

Skoro rozkład próbkowania średniej ma swój własny rozkład, ma też swoje własne odchylenie
standardowe — nazywane **błędem standardowym** (*standard error*, SE), żeby odróżnić je od
odchylenia standardowego pojedynczych obserwacji. Wzór teoretyczny:

```
SE = sigma / sqrt(n)
```

gdzie `sigma` to odchylenie standardowe populacji, a `n` to wielkość próbki.

In [ ]:
sigma_populacji = klienci["wydatki_miesieczne"].std(ddof=0)
n = 30

se_teoretyczny = sigma_populacji / np.sqrt(n)
se_symulowany = np.std(srednie_probkowe, ddof=1)  # z naszej symulacji z sekcji 2 (n=30)

print(f"SE teoretyczny (wzor): {se_teoretyczny:.2f}")
print(f"SE zaobserwowany w symulacji: {se_symulowany:.2f}")


Obie liczby powinny wyjść bardzo blisko siebie — nasza symulacja "z brutalnej siły"
(tysiące próbek) potwierdza gotowy wzór matematyczny, który za chwilę pozwoli nam liczyć
to bez symulowania niczego.

<a id="sec5"></a>
## 5. Jak wielkość próbki wpływa na precyzję

Wzór `SE = sigma / sqrt(n)` mówi coś bardzo konkretnego: żeby **zmniejszyć błąd
standardowy o połowę**, potrzeba **czterokrotnie większej** próbki (bo w mianowniku jest
pierwiastek). To ważna intuicja praktyczna — podwojenie wielkości próbki nie podwaja
precyzji, tylko poprawia ją o czynnik `sqrt(2) ≈ 1.41`.

In [ ]:
wielkosci_probek = [5, 10, 30, 50, 100, 200]
bledy_standardowe = [sigma_populacji / np.sqrt(n) for n in wielkosci_probek]

plt.figure(figsize=(8, 4))
plt.plot(wielkosci_probek, bledy_standardowe, marker="o")
plt.xlabel("Wielkosc probki (n)")
plt.ylabel("Blad standardowy (SE)")
plt.title("SE maleje wraz z n, ale coraz wolniej")
plt.show()


Krzywa spłaszcza się przy większych `n` — pierwsze kilkadziesiąt dodatkowych obserwacji
daje ogromną poprawę precyzji, a kolejne setki coraz mniejszą. To dokładnie ta sama
intuicja, do której wrócimy w Module 14 przy planowaniu wielkości próbki dla testu A/B:
"więcej danych zawsze pomaga, ale z malejącymi zyskami".

<a id="sec6"></a>
## 6. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- różnicę między populacją/parametrem a próbką/statystyką,
- rozkład próbkowania średniej — rozkład samych średnich z wielu próbek,
- Centralne Twierdzenie Graniczne: średnie próbkowe zbliżają się do rozkładu normalnego
  niezależnie od kształtu populacji, gdy `n` rośnie,
- błąd standardowy `SE = sigma / sqrt(n)` jako miarę precyzji średniej próbkowej,
- że poprawa precyzji wymaga nieproporcjonalnie większej próbki (pierwiastek w mianowniku).

> 📝 **Ćwiczenie 1: CTG dla innej kolumny**
>
> Powtórz symulację z sekcji 2 (5000 próbek po 30 obserwacji), ale dla kolumny `satysfakcja` zamiast `wydatki_miesieczne`. Narysuj histogram średnich próbkowych. Czy rozkład wygląda na zbliżony do normalnego, mimo że `satysfakcja` sama w sobie nie jest idealnie symetryczna?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
srednie_satysfakcji = [
    rng.choice(klienci["satysfakcja"], size=30, replace=False).mean()
    for _ in range(5000)
]

plt.figure(figsize=(8, 4))
plt.hist(srednie_satysfakcji, bins=40, edgecolor="white")
plt.xlabel("Srednia probki satysfakcji (n=30)")
plt.title("Rozklad probkowania sredniej satysfakcji")
plt.show()
```
</details>

> 📝 **Ćwiczenie 2: SE dla różnych kolumn**
>
> Policz teoretyczny błąd standardowy (`sigma / sqrt(n)`) dla `n=40` osobno dla kolumn `wydatki_miesieczne`, `wiek` i `satysfakcja`. Która kolumna ma największy błąd standardowy przy tej samej wielkości próbki? Co to mówi o jej rozproszeniu?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
for kolumna in ["wydatki_miesieczne", "wiek", "satysfakcja"]:
    sigma = klienci[kolumna].std(ddof=0)
    se = sigma / np.sqrt(40)
    print(f"{kolumna}: sigma={sigma:.2f}, SE(n=40)={se:.3f}")
```

Kolumna z najwiekszym rozproszeniem (najwiekszym sigma) bedzie miala tez najwiekszy SE przy tej samej wielkosci proby -- wieksza zmiennosc danych zrodlowych zawsze przeklada sie na mniej precyzyjna srednia probkowa.
</details>

> 📝 **Ćwiczenie 3: Ile razy większa próbka?**
>
> Załóżmy, że obecny SE dla `wydatki_miesieczne` przy `n=30` to pewna wartość X. Ile razy większą próbkę potrzeba, żeby zmniejszyć SE dokładnie o połowę? Sprawdź to obliczeniowo (policz SE dla `n=30` i dla `n=120`, porównaj stosunek).

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
se_30 = sigma_populacji / np.sqrt(30)
se_120 = sigma_populacji / np.sqrt(120)
print(f"SE przy n=30: {se_30:.3f}")
print(f"SE przy n=120: {se_120:.3f}")
print(f"Stosunek SE(30)/SE(120): {se_30 / se_120:.2f}")
```

4-krotne zwiekszenie n (30 -> 120) powinno dac dokladnie 2-krotne zmniejszenie SE, bo sqrt(4) = 2 -- to bezposrednia ilustracja wzoru z sekcji 5.
</details>

> 🔥 **Wyzwanie: CTG dla mediany zamiast średniej**
>
> CTG dotyczy w podstawowej formie średniej, ale można sprawdzić empirycznie, czy podobnie zachowuje się rozkład próbkowania MEDIANY. Powtórz symulację z sekcji 2, ale licząc medianę zamiast średniej dla każdej z 5000 próbek `wydatki_miesieczne` (n=30). Narysuj histogram median próbkowych i porównaj jego kształt z histogramem średnich próbkowych z sekcji 2 -- czy też wygląda w przybliżeniu na dzwonowaty?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
mediany_probkowe = [
    np.median(rng.choice(klienci["wydatki_miesieczne"], size=30, replace=False))
    for _ in range(5000)
]

plt.figure(figsize=(8, 4))
plt.hist(mediany_probkowe, bins=40, edgecolor="white")
plt.xlabel("Mediana probki (n=30)")
plt.title("Rozklad probkowania mediany")
plt.show()
```

Rozklad median probkowych tez zbliza sie do ksztaltu dzwonu (choc z reguly szerszego/mniej efektywnego niz dla sredniej) -- wersje CTG istnieja dla wielu statystyk, nie tylko dla sredniej, ale to srednia ma najprostsza, najbardziej znana teorie.
</details>

## Co dalej?

Teraz, gdy wiemy, że średnia próbkowa ma swój (w przybliżeniu normalny) rozkład i znamy
jej błąd standardowy, możemy w **Module 6** zbudować **przedział ufności** — zakres
wartości, w którym z określonym poziomem pewności mieści się prawdziwa, nieznana średnia
populacji.